In [ ]:
import os
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage

from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("groq_api_key")

# Initialize LLM
llm = ChatGroq(model="llama-3.3-70b-versatile", groq_api_key=api_key)
response = llm.invoke([HumanMessage(content="What is AI?")])
response.content

In [27]:
import os
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from dotenv import load_dotenv

load_dotenv()

# ── Config ──────────────────────────────────────────────────────────────────
PDF_PATH   = "/Users/ishant162/AI_Engineer_material/Agentic_AI_Six_month_roadmap/Notes_and_code/log/spruh73q.pdf"
LOG_PATH   = "/Users/ishant162/AI_Engineer_material/Agentic_AI_Six_month_roadmap/Notes_and_code/log/framework.log"
CHROMA_DIR = "./chroma_db"
EMBED_MODEL = "all-MiniLM-L6-v2"      # local, no API key needed

# ── LLM ─────────────────────────────────────────────────────────────────────
llm = ChatGroq(model="llama-3.3-70b-versatile", groq_api_key=os.getenv("groq_api_key"))

# ── Step 1: PDF Ingestion ────────────────────────────────────────────────────
def ingest_pdf():
    print("Loading PDF...")
    pages   = PyPDFLoader(PDF_PATH).load()                          # list of Document objects
    pages = pages[-200:]
    chunks  = RecursiveCharacterTextSplitter(
                  chunk_size=400, chunk_overlap=40
              ).split_documents(pages)
    
    embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)
    db = Chroma.from_documents(chunks, embeddings, persist_directory=CHROMA_DIR)
    db.persist()
    print(f"Ingested {len(chunks)} chunks from {len(pages)} pages → {CHROMA_DIR}")
    return db

# ── Step 2: Load existing ChromaDB (skip re-ingestion if already done) ───────
def load_db():
    embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)
    return Chroma(persist_directory=CHROMA_DIR, embedding_function=embeddings)

# ── Step 3: Log Analysis ─────────────────────────────────────────────────────
def analyse_log(log_path: str) -> str:
    log_text = open(log_path).read()

    messages = [
        SystemMessage(content=(
            "You are a log analysis assistant. "
            "Read the log and extract: (1) what failed, (2) the incorrect device slave address used. "
            "Reply in 2-3 sentences, be concise."
        )),
        HumanMessage(content=log_text),
    ]
    result = llm.invoke(messages)
    print("\n── Log Analysis ──")
    print(result.content)
    return result.content          # carries the bad address info forward

# ── Step 4: RAG – fetch correct address from manual ──────────────────────────
def rag_query(db, log_summary: str) -> str:
    # Build a targeted query from the log summary
    query = f"correct device address configuration: {log_summary}"
    docs  = db.similarity_search(query, k=1)
    context = "\n\n".join(d.page_content for d in docs)

    # return docs
    messages = [
        SystemMessage(content=(
            "You are a helpful assistant. Using only the provided context from the device manual, "
            "explain what the correct device address should be and why the one in the log was wrong."
        )),
        HumanMessage(content=f"Log summary:\n{log_summary}\n\nManual context:\n{context}"),
    ]
    result = llm.invoke(messages)
    print("\n── RAG Answer ──")
    print(result.content)
    return result.content

In [2]:
# ── Main ─────────────────────────────────────────────────────────────────────
    # Run ingest_pdf() once; after that comment it out and use load_db()
db = ingest_pdf()       # ← comment out after first run
# db = load_db()        # ← uncomment after first run

Loading PDF...


/var/folders/wc/rwpfcb3n6ysfjqkkml8ynztr0000gn/T/ipykernel_19528/3005388087.py:30: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)


Ingested 1077 chunks from 200 pages → ./chroma_db


/var/folders/wc/rwpfcb3n6ysfjqkkml8ynztr0000gn/T/ipykernel_19528/3005388087.py:32: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  db.persist()


In [16]:
log_summary = analyse_log(LOG_PATH)


── Log Analysis ──
The I2cRegisterReadTest failed. The incorrect device slave address used was 53, which is in decimal format and may be incorrect for the intended device. This address was used to read from the I2C bus with ID 0 and register 0.


The I2cRegisterReadTest failed. The incorrect device slave address used was 53, which is in decimal format and may be incorrect for the intended device. This address was used to read from the I2C bus with ID 0 and register 0.

In [28]:
rag_query(db, log_summary)


── RAG Answer ──
The correct device address should be 0x50 (or 80 in decimal), not 53. The reason for this is that the device manual specifically states that the ROM accesses the I2C EEPROM at I2C slave address 0x50. The address used in the log, 53, does not match this value, which is likely the cause of the I2cRegisterReadTest failure. The address 0x50 is in hexadecimal format, which is commonly used for I2C addresses, and it should be converted to decimal as 80, not 53.


'The correct device address should be 0x50 (or 80 in decimal), not 53. The reason for this is that the device manual specifically states that the ROM accesses the I2C EEPROM at I2C slave address 0x50. The address used in the log, 53, does not match this value, which is likely the cause of the I2cRegisterReadTest failure. The address 0x50 is in hexadecimal format, which is commonly used for I2C addresses, and it should be converted to decimal as 80, not 53.'